# 🏗️ Notebook 4 — Constructor Championship Model
**Formula 1 ML Analytics Project**

**Question answered**: *Will team X win the constructor championship?*

Model approach:
- Aggregate race-level data to **one row per constructor per season**
- Target: constructor championship final position
- Algorithm: XGBoost + Random Forest comparison
- Features: total points, avg finish, wins, podiums, DNFs, development trajectory


In [ ]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings("ignore")

sys.path.insert(0, "..")
DATA_PATH = "../data/processed/"
MODEL_PATH = "../models/"
os.makedirs(MODEL_PATH, exist_ok=True)

feat_file = os.path.join(DATA_PATH, "featured_df.csv")
if not os.path.exists(feat_file):
    print("⚠️  Run notebooks 01 and 02 first to generate featured_df.csv")
else:
    featured_df = pd.read_csv(feat_file, low_memory=False)
    print(f"✅ Loaded featured_df: {featured_df.shape}")


In [ ]:
from src.models import build_constructor_season_df, train_constructor_model, predict_constructor_championship

# Build team-season level dataset
constructor_df = build_constructor_season_df(featured_df)
print(f"Constructor season dataset: {constructor_df.shape}")
print(f"Season range: {constructor_df['year'].min()} – {constructor_df['year'].max()}")
print(f"\nSample:")
constructor_df.head(5)


In [ ]:
# Explore constructor championship data
if 'is_champion' in constructor_df.columns:
    print("Championship wins by constructor (top 15):")
    champ_wins = constructor_df[constructor_df['is_champion']==1].groupby('constructorName')['year'].count().sort_values(ascending=False)
    print(champ_wins.head(15).to_string())
    
    plt.figure(figsize=(12, 5))
    champ_wins.head(10).plot(kind='bar', color='#DC0000', alpha=0.8)
    plt.title('F1 Constructor Championship Wins (All Time)')
    plt.xlabel('Constructor')
    plt.ylabel('Championships')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# Train constructor championship model
print("Training constructor championship model...")
constructor_bundle = train_constructor_model(constructor_df)
print("✅ Constructor model trained.")

import joblib
joblib.dump(constructor_bundle, os.path.join(MODEL_PATH, "constructor_model.pkl"))
print(f"✅ Saved to {MODEL_PATH}constructor_model.pkl")


In [ ]:
# -----------------------------------------------------------------------
# USER-FACING PREDICTION: Will a team win the championship?
# -----------------------------------------------------------------------
teams_to_predict = [
    ("Red Bull", 2023),
    ("Mercedes", 2023),
    ("Ferrari", 2023),
    ("McLaren", 2024),
]

print("🏆 Constructor Championship Predictions")
print("="*70)
print(f"{'Team':<20} {'Year':>6} {'Pred Position':>15} {'Win Prob':>10} {'Pred Points':>12}")
print("-"*70)
for team, year in teams_to_predict:
    try:
        result = predict_constructor_championship(constructor_df, team=team, year=year, model=constructor_bundle)
        pos = result.get('predicted_position', 'N/A')
        prob = result.get('win_probability', 0)
        pts = result.get('predicted_points', 'N/A')
        print(f"{team:<20} {year:>6} {str(pos):>15} {prob:>10.1%} {str(pts):>12}")
    except Exception as e:
        print(f"{team:<20} {year:>6} {'Error':>15} — {e}")


In [ ]:
# Constructor points trajectory visualization
if 'year' in constructor_df.columns and 'total_points' in constructor_df.columns:
    top_teams = constructor_df.groupby('constructorName')['total_points'].sum().nlargest(8).index
    df_top = constructor_df[constructor_df['constructorName'].isin(top_teams)]
    
    plt.figure(figsize=(14, 6))
    for team in top_teams:
        team_data = df_top[df_top['constructorName']==team].sort_values('year')
        plt.plot(team_data['year'], team_data['total_points'], 
                marker='o', markersize=3, linewidth=2, label=team, alpha=0.85)
    plt.title('Constructor Season Points — Top Teams (All Time)', fontsize=13, fontweight='bold')
    plt.xlabel('Season')
    plt.ylabel('Total Season Points')
    plt.legend(loc='upper left', fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
